<a href="https://colab.research.google.com/github/sanketkandalkar46-del/dsa-pp/blob/main/AIML_GroupC_Assignment5_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIML Group C – Assignment 1: Clustering Techniques
**Course:** Artificial Intelligence & Machine Learning Lab (PCC-305-ITT) | **Class:** T.E. IT

**Problem Statement:** Implement K-Means Clustering and Hierarchical Clustering. Use the Elbow Method and Dendrogram for determining the optimal number of clusters. Analyze the generated clusters using a real-world dataset (Mall Customers).

## 0. Upload `Mall_Customers.csv` (Google Colab)
Run this cell and select your Mall Customers CSV. The file should be named `Mall_Customers.csv` (rename it if needed).

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

## Step 1: Data Exploration and Preprocessing

In [ ]:
# Importing the required libraries
import numpy as np                 # for Numeric Operations
import pandas as pd                # for Dataframe Operations
import matplotlib.pyplot as plt    # for Plotting and Visualization
import seaborn as sns

In [ ]:
# Loading the dataset
df = pd.read_csv("Mall_Customers.csv")

In [ ]:
# Reading the first n rows
df.head()

In [ ]:
# Checking the dimensions of the data frame
df.shape

In [ ]:
# Checking data types
df.dtypes

In [ ]:
# Renaming column name (only if the column is called 'Genre')
df.rename(columns={'Genre': 'Gender'}, inplace=True)

In [ ]:
# Printing information about a DataFrame
df.info()

In [ ]:
# Providing descriptive statistics for dataset
df.describe()

In [ ]:
# Checking for missing values in the data
print(df.isna().sum())
print("Total missing:", df.isna().sum().sum())

In [ ]:
# Checking duplicates
df.duplicated().sum()

In [ ]:
# Checking no. of unique values for each column
df.nunique()

In [ ]:
# Dropping features containing unique values
df.drop(columns=['CustomerID'], axis=1, inplace=True)
df.nunique()

## Step 2: Exploratory Data Analysis

In [ ]:
# Checking outliers
sns.boxplot(data=df)
plt.show()

In [ ]:
# Checking distribution of Annual Income
sns.histplot(df['Annual Income (k$)'], kde=True, stat='density')
plt.show()

In [ ]:
# Checking distribution of Age
sns.histplot(df['Age'], kde=True, stat='density')
plt.show()

In [ ]:
# Checking distribution of Spending Score
sns.histplot(df['Spending Score (1-100)'], kde=True, stat='density')
plt.show()

In [ ]:
# Checking distribution of categorical feature 'Gender'
sns.countplot(x='Gender', data=df)
plt.show()

In [ ]:
# Relationship between Annual Income, Spending Score and Gender
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)", hue="Gender")
plt.show()

## Step 3: Preprocess the data by scaling the features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
num_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
df[num_cols] = scaler.fit_transform(df[num_cols])
df.head(3)

## Step 4: Checking correlation among features

In [ ]:
df.corr(numeric_only=True)

## Step 5: Applying `LabelEncoder` for encoding binary categories in 'Gender' column

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])
df.head(3)

## Step 6: Finding optimal number of clusters – Elbow Method and Silhouette Score

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.cluster import KMeans

wcss_list = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df)
    wcss_list.append(kmeans.inertia_)

wcss_list

In [ ]:
# Elbow plot
sns.lineplot(x=list(range(2, 11)), y=wcss_list, marker='o')
plt.xlabel("Number of clusters (k)")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()

**Elbow Method:** plots WCSS (Within-Cluster Sum of Squares) against the number of clusters *k*. The "elbow", where the decrease sharply slows down, suggests the optimal k (here k = 5).

**Silhouette Score** ranges from -1 to +1: +1 = well clustered, 0 = on/near the boundary between clusters, -1 = misclassified.

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df)
    labels = kmeans.predict(df)

    # Calculate Silhouette Score
    score = silhouette_score(df, labels)
    silhouette_scores.append(score)

sns.lineplot(x=list(range(2, 11)), y=silhouette_scores, marker='o')
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs k")
plt.show()

## Step 7: Applying K-Means clustering algorithm

In [ ]:
# Features used for clustering (before adding any cluster label columns)
X = df[['Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']].copy()

num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)
df.head(3)

## Step 8: Visualize the resulting clusters

In [ ]:
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue="cluster", data=df, palette='tab10')
plt.title("K-Means Clusters (k = 5)")
plt.show()

## Step 9: Hierarchical Clustering (Agglomerative) and Dendrogram

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
# Compute the linkage matrix (Ward method) on the feature columns
matrix = linkage(X, method='ward')

In [ ]:
# Plot the dendrogram
plt.figure(figsize=(10, 7))
dendrogram(matrix, orientation='top', distance_sort='descending', show_leaf_counts=True)
plt.title('Dendrogram')
plt.xlabel('Sample index')
plt.ylabel('Distance')
plt.show()

In [ ]:
# Apply Agglomerative Clustering
# Set the number of clusters, for example, k=4 (as in the manual)
num_clusters = 4
agg_clustering = AgglomerativeClustering(n_clusters=num_clusters, linkage='ward')
clusters = agg_clustering.fit_predict(X)

In [ ]:
# Add cluster labels to the data
df['Cluster1'] = clusters
df.head(3)

In [ ]:
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster1', palette='tab10')
plt.title("Hierarchical (Agglomerative) Clusters")
plt.show()

## Conclusion
The clustering analysis successfully segmented customers into distinct groups, revealing valuable insights into their spending behavior.

## Short Answer Questions

**1. What is the purpose of scaling the data before applying clustering algorithms?**
Clustering uses distance. Features with large ranges (e.g. Annual Income) would dominate small ones (e.g. Age). Scaling puts all features on a comparable scale so each contributes fairly.

**2. How does the Elbow Method help in determining the optimal number of clusters for K-Means?**
It plots WCSS for different values of k. WCSS falls as k increases; the point where the drop becomes slow (the "elbow") is taken as the optimal k.

**3. What is the significance of the Silhouette Score?**
It measures how similar a point is to its own cluster compared with other clusters (range -1 to +1). A higher score means better-defined, well-separated clusters; the k with the highest score is a good choice.

**4. How does hierarchical clustering differ from K-Means?**
K-Means needs k in advance, uses centroids and gives flat clusters. Hierarchical clustering builds a tree (hierarchy) by merging or splitting clusters and does not need k beforehand; k is chosen by cutting the dendrogram.

**5. What does the dendrogram represent?**
A tree diagram showing the order and distance at which points/clusters are merged. The lower the merge height, the more similar the groups. Cutting it at a height gives the number of clusters.

**6. Purpose of the linkage function and its use in the code**
`linkage(X, method='ward')` computes the hierarchical clustering linkage matrix: it repeatedly merges the closest clusters and records the merge distances. Ward linkage merges the pair of clusters that gives the smallest increase in total within-cluster variance. The matrix is passed to `dendrogram()` to draw the tree.